# nb16 — Tier 2.5: Hybrid HDC-RWKV (Breaking the Binary Ceiling)

### Background

Per F13 in [findings.md](../docs/findings.md), pure-binary HDC-RWKV has a
**scale-independent ceiling at BPC ~2.93**. Doubling, quadrupling, and
octupling the model didn't move it. The bottleneck was the bipolar
prototype-similarity output.

### What's new in Tier 2.5

Same architecture as HDC-RWKV with ONE change: the output projection
(prototype matrix) is **continuous** (int8-quantized at deployment) rather
than bipolar. Everything else stays binary:

| Component | Pure binary (Tier 3) | **Hybrid (Tier 2.5)** |
|---|---|---|
| vocab_hv | bipolar (1 bit/dim) | bipolar (unchanged) |
| decay_mask | bipolar (1 bit/dim) | bipolar (unchanged) |
| State recurrence | continuous tanh | continuous tanh (unchanged) |
| **Prototype output** | **bipolar** | **int8 (real matmul)** |
| Storage at V=256, d=512 | ~16 KB | **~148 KB** |
| Deployment target | 6502 (32 KB EEPROM) | **ESP32 flash (4 MB)** |

### Target outcome

BPC < 2.5 (and ideally < 2.2). If we achieve this, the architecture is
validated at the ESP32 deployment scale and the paper has Tier 2.5 as the
middle deployment between teacher (untouchable) and 6502 (extreme limit).


## Cell 1 — Setup (clone latest wozformer with HDCRWKVHybrid class)

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir('/kaggle/working')
REPO_URL = 'https://github.com/elixpo/wozformer.git'
subprocess.run(['rm', '-rf', '/kaggle/working/wozformer'], check=True)
subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/wozformer'], check=True)
WOZFORMER_PATH = Path('/kaggle/working/wozformer')
os.chdir(WOZFORMER_PATH)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

# Force-evict cached modules
for m in list(sys.modules):
    if m.startswith('wozformer'):
        del sys.modules[m]

import wozformer as wz
import torch
import math
import matplotlib.pyplot as plt

print(f'wozformer {wz.__version__}, device: {wz.utils.get_device()}')
assert hasattr(wz.models, 'HDCRWKVHybrid'), 'HDCRWKVHybrid not exported — check the repo'


## Cell 2 — Hyperparameters

V=256 and d=512 chosen to stay within HDC capacity (V/(d/log(d)) ≈ 4)
AND keep deployment ~150 KB for clean ESP32 fit.


In [ ]:
VOCAB_SIZE = 256
D          = 512
N_LAYERS   = 1     # single layer matches the working point from nb12c
BLOCK_SIZE = 64

BATCH_SIZE = 32
LR         = 3e-3
N_STEPS    = 20000
EVAL_EVERY = 500
SEED       = 1337

RUN_DIR = Path('/kaggle/working/runs')
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_PT  = RUN_DIR / 'tier2_5_hybrid.pt'
OUT_BIN = RUN_DIR / 'tier2_5_hybrid.bin'
BPE_JSON = RUN_DIR / 'bpe_256.json'


## Cell 3 — Corpus + BPE

In [ ]:
wz.utils.set_seed(SEED)
device = wz.utils.get_device()

text = wz.data.load_corpus(WOZFORMER_PATH / 'data' / 'tinyshakespeare.txt')
tok = wz.tokenizer.BPETokenizer.train(text, vocab_size=VOCAB_SIZE)
tok.save(BPE_JSON)

ids = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = wz.data.split_train_val(ids)
print(f'tokens: train {len(train_data):,} / val {len(val_data):,}')


## Cell 4 — Build the hybrid model

In [ ]:
cfg = wz.config.HDCRWKVHybridConfig(
    vocab_size=VOCAB_SIZE,
    d=D,
    n_layers=N_LAYERS,
    block_size=BLOCK_SIZE,
)
model = wz.models.HDCRWKVHybrid(cfg).to(device)

n_train = wz.utils.count_params(model)
n_bytes = model.deployment_bytes()
print(f'trainable params: {n_train:,}')
print(f'deployment bytes: {n_bytes:,} ({n_bytes/1024:.1f} KB)')
print(f'  breakdown: vocab+decay bipolar={(VOCAB_SIZE*D + N_LAYERS*D)//8:,} B,'
      f' prototype int8={VOCAB_SIZE*D:,} B')

# Sanity
xb, yb = wz.data.make_batch(train_data, BATCH_SIZE, BLOCK_SIZE, device)
with torch.no_grad():
    _, loss = model(xb, yb)
print(f'init loss: {loss.item():.4f} (expect ~{math.log(VOCAB_SIZE):.4f})')


## Cell 5 — Train

~30-60 min on a T4. The trainer tracks:
- val(soft): STE + fp32 prototype
- val(hard): real .sign() + int8-quantized prototype (what deployment runs)

Healthy outcome: val(hard) descends below 4.0 nats by step 5000 and continues
below 3.5 by step 15000. If val(hard) plateaus near 5.0, the int8 quantization
is destroying signal — we'd need to retain fp16 prototype instead.


In [ ]:
train_cfg = wz.config.TrainConfig(
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE,
    lr=LR,
    n_steps=N_STEPS,
    eval_every=EVAL_EVERY,
    seed=SEED,
    weight_decay=0.0,
)
history, best = wz.trainer.train(
    model, train_data, val_data, train_cfg,
    device=device, eval_hard=True,
)
print(f'\nbest HARD val: {best["val"]:.4f} at step {best["step"]}')


## Cell 6 — Loss curves + BPC

In [ ]:
steps = [h[0] for h in history]
trains = [h[1] for h in history]
vals_soft = [h[2] for h in history]
vals_hard = [h[3] for h in history]

plt.figure(figsize=(10, 4.5))
plt.plot(steps, trains, label='train', alpha=0.6)
plt.plot(steps, vals_soft, label='val(soft)', alpha=0.8)
plt.plot(steps, vals_hard, label='val(hard, int8 deploy)', linewidth=2, color='C2')
plt.axhline(2.93 * 2.14 * math.log(2), color='red', linestyle='--', alpha=0.5,
            label=f'nb12c ceiling (val ~4.35)')
plt.xlabel('step'); plt.ylabel('val (nats/token)')
plt.title('Hybrid HDC-RWKV — does int8 prototype break the binary ceiling?')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# BPC
sample = val_data[:5000].tolist()
avg_cpt = wz.metrics.avg_chars_per_token(tok, sample)
bpc = wz.metrics.bits_per_char(best['val'], avg_cpt)
print(f'avg chars/token: {avg_cpt:.2f}')
print(f'best HARD val:   {best["val"]:.4f} nats/token')
print(f'Tier 2.5 BPC:    {bpc:.4f}')
print()
print('Reference:')
print(f'  Teacher (fp32 transformer): BPC 2.04')
print(f'  nb12c (pure binary, 16 KB): BPC 2.93  <-- the ceiling we tried to break')
if bpc < 2.3:
    print(f'\n  >> CEILING BROKEN. Tier 2.5 reaches BPC {bpc:.2f}, near teacher quality.')
elif bpc < 2.7:
    print(f'\n  >> Significant improvement (BPC {bpc:.2f} vs 2.93 baseline).')
elif bpc < 2.9:
    print(f'\n  >> Marginal improvement. The binary recurrence may still be limiting.')
else:
    print(f'\n  >> No improvement — the bottleneck is elsewhere.')


## Cell 7 — Generate samples and judge by eye

Use the same 5 prompts and seeds as Tier 2 (nb15) so output is directly comparable.


In [ ]:
prompts = ['king', 'romeo', 'my lord,', 'queen elizabeth:', 'to be or not']
seeds = [1337, 42, 7, 99, 2024]

for prompt, seed in zip(prompts, seeds):
    print(f'\n===== {prompt!r}  seed={seed} =====')
    out = wz.generate.generate(
        model, tok, prompt=prompt, max_new_tokens=120,
        block_size=BLOCK_SIZE, temperature=0.7, top_k=10,
        seed=seed, device=device, use_hard=True,
    )
    print(out)


## Cell 8 — Save artifacts + export int8-quantized binary

Binary format (Tier 2.5 v1):

```
offset  size      contents
------  --------  ----------------------------------------
0       4         magic 'WHBR' (Hybrid Binary Recurrent)
4       1         version (1)
5       2         vocab_size (uint16 LE)
7       2         d/8       (uint16 LE)
9       1         block_size
10      1         n_layers
11      4         log_temp (float32 LE)
15      4         prototype_scale (float32 LE)
19      1         reserved

20      V*d/8     vocab_hv  (packed bipolar bits)
...     L*d/8     decay_mask (packed bipolar bits) per layer
...     V*d       prototype matrix (int8 signed bytes)
```


In [ ]:
import struct
import numpy as np

def pack_bits(t):
    return np.packbits((t > 0).to(torch.uint8).cpu().numpy(), axis=-1, bitorder='big')

vocab_packed = pack_bits(model.vocab_hv_c.data)
decay_packed = [pack_bits(dm.data.unsqueeze(0)).squeeze(0) for dm in model.decay_masks_c]
proto_q, proto_scale = model.quantize_prototype_int8()

buf = bytearray()
buf += b'WHBR'                                     # magic for HybridBinaryRecurrent
buf += bytes([1])                                  # version
buf += struct.pack('<H', VOCAB_SIZE)
buf += struct.pack('<H', D // 8)
buf += bytes([BLOCK_SIZE, N_LAYERS])
buf += struct.pack('<f', model.log_temp.item())
buf += struct.pack('<f', proto_scale)
buf += bytes(1)                                    # reserved

buf += vocab_packed.tobytes()
for dp in decay_packed:
    buf += dp.tobytes()
buf += proto_q.cpu().numpy().tobytes()             # int8 prototype

OUT_BIN.write_bytes(buf)
print(f'saved binary → {OUT_BIN}  ({len(buf):,} bytes = {len(buf)/1024:.1f} KB)')

torch.save({
    'config': cfg.__dict__,
    'model_state': model.state_dict(),
    'history': history,
    'best_val_hard': best['val'],
    'best_step': best['step'],
    'deploy_bytes': n_bytes,
    'prototype_scale': proto_scale,
    'tier': 'tier2.5_hybrid_int8_prototype',
}, OUT_PT)
print(f'saved checkpoint → {OUT_PT}  ({OUT_PT.stat().st_size/1024:.1f} KB)')

print(f'\nDownload from /kaggle/working/runs/ before closing kernel:')
print(f'  - {OUT_BIN.name}')
print(f'  - {OUT_PT.name}')
print(f'  - {BPE_JSON.name}')


## Post-mortem

After running, paste the final BPC + generation samples back. We'll add F14 to
findings.md with the result:

- BPC < 2.3 → F14: int8 prototype breaks the binary recurrence ceiling
- BPC 2.3–2.7 → F14: int8 prototype provides significant relief but not full break
- BPC > 2.7 → F14: bottleneck is elsewhere (probably recurrence itself)
